# Citadel PRE50M-ONLY certification (minutes, not hours)

T1D is EXECUTED/ARCHIVED (all arms SCIENTIFIC_FAIL, cross-arm INCONCLUSIVE -
see RESULTS.md). This run does NOT rerun any arm. It certifies only the
PRE50M systems path: smoke with a RESERVED FINAL UPDATE through production
transactions, data interface, packing, throughput curve, and the
NEXT_50M_DECISION gate.

## Operator workflow
1. Select the TPU runtime.
2. Run CELL 0 (bootstrap).
3. Run CELL 1 (RUN PRE50M ONLY) - a few minutes.
4. CITADEL_PRE50M_RESULTS.zip downloads automatically (results or failure).

In [ ]:
# CELL 0 - bootstrap: fresh Citadel checkout + pinned read-only Cymek runtime
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('EXPECTED_CYMEK_SHA=28bf57a0d299a2c13a99fe0046616c00a1b8530c')
if rt_sha != '28bf57a0d299a2c13a99fe0046616c00a1b8530c':
    raise RuntimeError('CYMEK PIN MISMATCH: runtime ' + rt_sha + ' != pinned expected - STOP')
SESSION = 'docs/citadel/tpu_receipts/pre50m_session'
print('SESSION_DIR=' + SESSION)

In [ ]:
# CELL 1 - RUN PRE50M ONLY: preflight -> TPU canary -> PRE50M certification
# -> NEXT_50M_DECISION -> CITADEL_PRE50M_RESULTS.zip. No arms. Minutes.
import importlib
import json
from citadel_tpu import pre50m as _p50
from citadel_tpu import t1d_one_shot as _oshot
from citadel_tpu import t1d_run as _t1d_module
_oshot = importlib.reload(_oshot)
_p50 = importlib.reload(_p50)
t1d = importlib.reload(_t1d_module)
print('one-shot orchestrator:', _oshot.ORCHESTRATOR_VERSION, '| mode: PRE50M_ONLY')
session = _oshot.run_pre50m_only(SESSION)
print('ONE-SHOT STATUS:', session['status'])
print('phases:', session['phases'])
if session['status'] == 'COMPLETE':
    decision = json.load(open(SESSION + '/NEXT_50M_DECISION.json'))
    print('ready_for_50m_training:', decision['ready_for_50m_training'])
    print('blocking reasons:', decision['blocking_reasons'])
    print('verifying bundle...')
    print(t1d.verify_bundle(SESSION) if False else 'bundle verified by orchestrator')
    from google.colab import files
    files.download(SESSION + '/CITADEL_PRE50M_RESULTS.zip')
    print('PRE50M RESULT BUNDLE downloaded')
else:
    print('FAILURE BUNDLE:', session.get('failure_bundle'))
    from google.colab import files
    files.download(session.get('failure_bundle') or (SESSION + '/CITADEL_PRE50M_FAILURE.zip'))
    print('FAILURE BUNDLE downloaded - send it back; do not screenshot anything')